# 02 — Feature Engineering Experiments

This notebook validates the preprocessing pipeline, inspects split shapes, fits a quick
LightGBM model, and verifies that lag-168 appears in the top feature importances.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import lightgbm as lgb
import plotly.express as px

from src.data.download import download_data
from src.data.preprocess import run_preprocessing

download_data()
train_df, val_df, test_df = run_preprocessing()

## 1. Split Shapes

In [ ]:
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name:5s}: {df.shape[0]:6,} rows × {df.shape[1]} cols')
    
total = len(train_df) + len(val_df) + len(test_df)
print(f'\nRatios — train: {len(train_df)/total:.1%}  val: {len(val_df)/total:.1%}  test: {len(test_df)/total:.1%}')

## 2. Quick LightGBM Fit (sanity check)

In [ ]:
TARGET = 'OT'
feature_cols = [c for c in train_df.columns if c != TARGET]

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

params = dict(
    n_estimators=300, learning_rate=0.05,
    num_leaves=31, verbosity=-1,
    random_state=42,
)

model = lgb.train(
    params,
    dtrain,
    valid_sets=[dval],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(50)],
)

from sklearn.metrics import mean_squared_error
preds = model.predict(X_val)
rmse  = np.sqrt(mean_squared_error(y_val, preds))
print(f'\nQuick-fit val RMSE: {rmse:.4f}')

## 3. Feature Importance Bar Chart

In [ ]:
importance_df = pd.DataFrame({
    'feature': model.feature_name(),
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

fig = px.bar(
    importance_df,
    x='importance', y='feature',
    orientation='h',
    title='LightGBM Feature Importance (gain)',
    color='importance',
    color_continuous_scale='Blues',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 4. Confirm lag-168 in Top 5

In [ ]:
top5 = importance_df.head(5)
print('Top 5 features by gain:')
print(top5.to_string(index=False))

assert 'OT_lag_168' in top5['feature'].values, (
    f'OT_lag_168 not in top 5! Got: {top5["feature"].tolist()}'
)
print('\n✓ OT_lag_168 confirmed in top 5 features.')

## 5. Temporal Order Check

In [ ]:
assert train_df.index.max() < val_df.index.min(), 'Train/val overlap!'
assert val_df.index.max() < test_df.index.min(), 'Val/test overlap!'

print('Temporal ordering check passed:')
print(f'  Train: {train_df.index.min()} → {train_df.index.max()}')
print(f'  Val:   {val_df.index.min()} → {val_df.index.max()}')
print(f'  Test:  {test_df.index.min()} → {test_df.index.max()}')